# Evaluation of various sentence embedding training

Biblical-reuse benchmark over a fixed Vulgate corpus.

We evaluate each encoder on two STS-based tasks:
1. Text reuse detection (full dataset): max STS over all verses -> AP / PR-AUC and ROC curve.
2. IR in Vulgate (reuse-only): top-1 verse assignment -> match rate against gold verse sets.

Outputs are saved as LaTeX tables in `results/` and ROC curves as PDFs.

In [1]:
# Install FAISS once if missing.
import importlib.util, subprocess, sys
if importlib.util.find_spec('faiss') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-cpu'])

In [2]:
# Install eval/plotting deps once if missing.
import importlib.util, subprocess, sys
if importlib.util.find_spec('sklearn') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn'])
if importlib.util.find_spec('matplotlib') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'matplotlib'])

In [3]:
import gc, os, re, json, time
import unicodedata
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Set, Tuple
import unicodedata as _ud

import faiss
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, BertConfig, BertModel
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer
from sentence_transformers import util as st_util
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_curve, auc

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| torch:', torch.__version__, '| faiss has gpu:', faiss.get_num_gpus() if hasattr(faiss, 'get_num_gpus') else 'n/a')

/home/tdelaselle/anaconda3/envs/patrisbert/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda | torch: 2.11.0+cu130 | faiss has gpu: 0


/tmp/ipykernel_30873/2821570140.py:19: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import InformationRetrievalEvaluator


In [4]:
# Configuration
ROOT = Path('.').resolve()
BIBLES_DIR = ROOT / 'data' / 'bibles'
VG_PATH = BIBLES_DIR / 'VG.tsv' # or BIBLES_DIR / 'LXX+NTG.tsv'
REUSES_DIR = ROOT / 'data' / 'reuses_test_set'
# Select exactly one patristic-work test file for this evaluation.
REUSE_FILE = 'Aug_trin_sentences.tsv'
RESULTS_DIR = ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = ROOT / 'embeddings'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
MAX_SEQ_LEN = 256
BATCH_SIZE = 64
ALLOW_REMOTE = True

## 1. Load the Bible index and reuse test set

Bible TSVs are read from `data/bibles/`. The selected patristic-work test file is read from `data/reuses_test_set/`; its filename is added as `source_file` because the files contain only `sentence_index`, `text`, `label`, and `reference` columns.

In [5]:
vg = pd.read_csv(VG_PATH, sep='\t', dtype=str, keep_default_na=False)
vg['text'] = vg['text'].str.strip()
# vg = vg[vg['text'].str.split().str.len() > 4].reset_index(drop=True)
vg_refs = vg['reference'].tolist()
vg_texts = vg['text'].tolist()
ref_to_vg_idx: Dict[str, int] = {r: i for i, r in enumerate(vg_refs)}
print(f'Vulgate index: {len(vg_refs):,} verses')
vg.head(3)

Vulgate index: 37,209 verses


,reference,book,chapter,verse,suffix,text
0,1 Ch 10:1,1 Chronicles,10,1,,Philisthim autem pugnabant contra Israhel fuge...
1,1 Ch 10:10,1 Chronicles,10,10,,arma autem eius consecraverunt in fano dei sui...
2,1 Ch 10:11,1 Chronicles,10,11,,hoc cum audissent viri Iabesgalaad omnia scili...


In [6]:
reuse_path = REUSES_DIR / REUSE_FILE
if not reuse_path.is_file():
    available_files = sorted(p.name for p in REUSES_DIR.glob('*_sentences.tsv'))
    raise FileNotFoundError(
        f'Reuse test file not found: {reuse_path}. Available files: {available_files}'
    )

reuses = pd.read_csv(reuse_path, sep='\t', dtype=str, keep_default_na=False)
expected_columns = ['sentence_index', 'text', 'label', 'reference']
if list(reuses.columns) != expected_columns:
    raise ValueError(f'Unexpected columns in {reuse_path.name}: {list(reuses.columns)}')
reuses.insert(0, 'source_file', reuse_path.name)
print('reuse file:', reuse_path.name)
print('rows:', len(reuses))
print(reuses['label'].value_counts())
reuses.head(3)

reuse file: Aug_trin_sentences.tsv
rows: 4410
label
NaR           3116
Occurrence    1196
NaBR            98
Name: count, dtype: int64


,source_file,sentence_index,text,label,reference
0,Aug_trin_sentences.tsv,0,domino beatissimo et sincerissima caritate uen...,NaR,o
1,Aug_trin_sentences.tsv,1,de trinitate quae deus summus et uerus est lib...,NaR,o
2,Aug_trin_sentences.tsv,2,omiseram quippe hoc opus posteaquam comperi pr...,NaR,o


## 2. Reference parsing — expand verse ranges and align with VG

Patristic references like `1 Ch 16:41-42` need to be expanded into a set of canonical verse keys (`1 Ch 16:41`, `1 Ch 16:42`). A reuse is correctly retrieved if **any** verse from its expanded gold set is in the top-k.

In [7]:
REF_RE = re.compile(r'^\s*(?P<book>.+?)\s+(?P<chap>\d+):(?P<vs>\d+)(?:-(?P<ve>\d+))?\s*$')
DOTLIST_RE = re.compile(r'^\s*(?P<book>.+?)\s+(?P<chap>\d+):(?P<vslist>\d+(?:\.\d+)+)\s*$')

def parse_reference(raw: str) -> List[str]:
    """Return the list of canonical verse keys covered by a reference string."""
    if not raw:
        return []
    raw = raw.strip().rstrip(';').strip()
    if raw in ('', 'o', ';'):
        return []
    if ';' in raw:
        parts = [p.strip() for p in raw.split(';')]
        out: List[str] = []
        for part in parts:
            if part in ('', 'o', ';'):
                continue
            out.extend(parse_reference(part))
        return out
    m = DOTLIST_RE.match(raw)
    if m:
        book = m.group('book').strip()
        chap = int(m.group('chap'))
        verses = [int(v) for v in m.group('vslist').split('.')]
        return [f'{book} {chap}:{v}' for v in verses]
    m = REF_RE.match(raw)
    if not m:
        return []
    book = m.group('book').strip()
    chap = int(m.group('chap'))
    vs = int(m.group('vs'))
    ve = int(m.group('ve')) if m.group('ve') else vs
    if ve < vs:
        ve = vs
    return [f'{book} {chap}:{v}' for v in range(vs, ve + 1)]

# sanity check
# for r in ['Ml 4:2', '1 Ch 16:41-42', 'Si 14:5-6 ;', 'o', '', 'Jn 1:2 ; Jn 1:3', 'Jn 17:11.9.6', "Ex 20:13 ; Ex 20:15"]:
#     print(repr(r), '->', parse_reference(r))

In [8]:
# For each unique patristic sentence (source_file, sentence_index) collect:
#   - text
#   - set of reuse types attached to it
#   - set of gold VG indices it points to (after parsing + matching VG)
#   - is_nar flag (true iff every row for that sentence is NaR)
sent_text: Dict[Tuple[str, str], str] = {}
sent_types: Dict[Tuple[str, str], Set[str]] = defaultdict(set)
sent_gold: Dict[Tuple[str, str], Set[int]] = defaultdict(set)
sent_is_nar: Dict[Tuple[str, str], bool] = {}

matched_refs = 0
skipped_refs = 0
for row in reuses.itertuples(index=False):
    key = (row.source_file, row.sentence_index)
    sent_text[key] = row.text.strip()
    if row.label == 'NaR' or row.label == 'NaBR':
        sent_is_nar.setdefault(key, True)
        continue
    sent_is_nar[key] = False
    sent_types[key].add(row.label)
    expanded = parse_reference(row.reference)
    found = False
    for r in expanded:
        idx = ref_to_vg_idx.get(r)
        if idx is not None:
            sent_gold[key].add(idx)
            found = True
    if found:
        matched_refs += 1
    else:
        skipped_refs += 1
        print(f'Warning: could not match any VG reference for "{row.reference}" in sentence {key}')

n_total_sentences = len(sent_text)
n_nar_sentences = sum(1 for v in sent_is_nar.values() if v)
n_reuse_sentences = n_total_sentences - n_nar_sentences
n_reuse_with_gold = sum(1 for k in sent_gold if sent_gold[k])
print(f'Unique sentences: {n_total_sentences:,}')
print(f'  NaR-only sentences: {n_nar_sentences:,}')
print(f'  Reuse sentences: {n_reuse_sentences:,}  ({n_reuse_with_gold:,} aligned to VG)')
print(f'Reuse rows: matched_refs={matched_refs:,}  skipped_refs={skipped_refs:,}')

Unique sentences: 4,410
  NaR-only sentences: 3,214
  Reuse sentences: 1,196  (1,189 aligned to VG)
Reuse rows: matched_refs=1,189  skipped_refs=7


### Greek normalisation

In [9]:
# === Greek text normalisation before encoding ================================
# Apply the unicode / case / elision normalisation that each Greek encoder
# actually expects (verified by inspecting their tokenizer.json):
#
#   'greberta'  -> bowphs/GreBerta, bowphs/SPhilBerta, models/GreBERTa/*
#                  Byte-level BPE, NO normalizer. CASE-SENSITIVE and accent-aware,
#                  trained on precomposed text -> needs NFC (NFD fragments tokens).
#                  Keep case + accents; only NFC + unify apostrophes.
#
#   'agbert'    -> pranaydeeps/Ancient-Greek-BERT
#                  WordPiece + BertNormalizer(lowercase=True, strip_accents=None
#                  -> behaves as True): the tokenizer lowercases AND strips accents
#                  internally. The one thing text MUST get right is the elision
#                  mark: ᾽U+1FBD / ᾿U+1FBF / ʼU+02BC are OOV -> [UNK]. We mirror the
#                  model (deaccent + lowercase) and unify apostrophes. Feed NFC.
#
# Set GREEK_NORM_TARGET to the family you are about to encode with, then run this
# cell before the encoding loop. NOTE: cached embeddings in CACHE_DIR are keyed by
# model name only, so delete the relevant *_vg.npy / *_queries.npy if you change
# the normalisation and want to re-encode.

GREEK_NORM_TARGET = 'greberta'   # 'greberta' | 'agbert' | None (leave text as-is)

# Spacing apostrophe / elision / koronis variants -> one canonical mark.
# (’ U+2019 is in-vocab for Ancient-Greek-BERT and unambiguous for byte-level BPE.)
_CANON_APOSTROPHE = '’'
_APOSTROPHE_VARIANTS = "\u0027\u2018\u2019\u02BC\u02B9\u1FBD\u1FBF\u0060\u00B4"

def _unify_apostrophes(text: str) -> str:
    for ch in _APOSTROPHE_VARIANTS:
        if ch != _CANON_APOSTROPHE:
            text = text.replace(ch, _CANON_APOSTROPHE)
    return text

def _strip_accents(text: str) -> str:
    text = _ud.normalize('NFD', text)
    return ''.join(c for c in text if not _ud.combining(c))

def normalize_greek(text: str, target: str) -> str:
    if not target:
        return text
    text = _ud.normalize('NFC', text)     # compose accents/breathings onto letters
    text = _unify_apostrophes(text)       # spacing elision marks -> canonical
    if target == 'agbert':
        # mirror BertNormalizer: deaccent + lowercase, re-compose to NFC
        text = _ud.normalize('NFC', _strip_accents(text).lower())
    elif target != 'greberta':
        raise ValueError(f"unknown GREEK_NORM_TARGET: {target!r}")
    return text

# Text normalization is applied in the query-preparation cell, after both
# the Bible and reuse sentence lists have been created.


## 3. Model registry

> **Warning — model selection:** Only the models uncommented in the registry below are evaluated. The active set, local paths, and remote Hugging Face model revisions are part of the experimental configuration; record them for reproducibility before releasing results.

Two loading paths:
- **`sbert`** — the directory has `modules.json` and a `1_Pooling/` folder; load with `SentenceTransformer`.
- **`hf`** — only HuggingFace transformer weights + tokenizer. Load with `AutoModel` and apply mean-pooling manually (matches what the DiffCSE / InfoCSE training scripts do).

In [10]:
# Stable ordering: array indexing for FAISS lookups
all_keys = list(sent_text.keys())
all_texts = [sent_text[k] for k in all_keys]

# Keep pristine copies so normalization is idempotent and rerunnable.
if 'vg_texts_raw' not in globals():
    vg_texts_raw = list(vg_texts)
if 'all_texts_raw' not in globals():
    all_texts_raw = list(all_texts)
vg_texts = [normalize_greek(t, GREEK_NORM_TARGET) for t in vg_texts_raw]
all_texts = [normalize_greek(t, GREEK_NORM_TARGET) for t in all_texts_raw]
print(f"Greek normalisation target: {GREEK_NORM_TARGET!r}")
print(f"  normalised {len(vg_texts):,} VG texts and {len(all_texts):,} query texts")

is_nar = np.array([sent_is_nar[k] for k in all_keys], dtype=bool)
gold_lists = [sent_gold[k] for k in all_keys]   # set[int] of VG indices, possibly empty
type_lists = [sent_types[k] for k in all_keys]  # set[str] of reuse types, empty for NaR
print('queries to embed:', len(all_texts))

# Length filter: sentences with fewer than 5 words are systematically
# predicted as 'not a reuse' downstream (forced-negative). Mirrors the >4-word
# filter already applied to the Vulgate index.
short_query_mask = np.array([len(t.split()) < 5 for t in all_texts], dtype=bool)
print('short queries (<5 words, forced not-a-reuse):', int(short_query_mask.sum()))

Greek normalisation target: 'greberta'
  normalised 37,209 VG texts and 4,410 query texts
queries to embed: 4410
short queries (<5 words, forced not-a-reuse): 0


In [ ]:
MODELS = [

    {'name': 'LaBerta-CSE_trin',   'path': 'models/LaBERTa/LaBERTaSimCSE_trin',   'type': 'sbert'},
    {'name': 'LaBERTa_TSDAE',    'path': 'models/LaBERTa/LaBERTaTSDAE',    'type': 'sbert'},
    # {'name': 'LaBerta-CSE_cfaust',   'path': 'models/LaBERTa/LaBERTaSimCSE_cfaust',   'type': 'sbert'},
    # {'name': 'LaBerta-CSE_trin_HTR',   'path': 'models/LaBERTa/LaBERTaSimCSE_trin_HTR',   'type': 'sbert'},
    # {'name': 'GreBERTa-CSE',   'path': 'models/GreBERTa/GreBERTaSimCSE',   'type': 'sbert'},    
    # {'name': 'Logion-CSE',   'path': 'models/Logion/LogionSimCSE',   'type': 'sbert'},  
    
    # Models from Hugging Face hub (remote=True means we won't check local path existence)
    # {'name': 'LatinBERT', 'path': 'ashleygong03/bamman-burns-latin-bert', 'type': 'latinbert', 'remote': True},
    {'name': 'LaBerta', 'path': 'bowphs/LaBerta', 'type': 'hf', 'remote': True},
    {'name': 'SPhilBerta', 'path': 'bowphs/SPhilBerta', 'type': 'hf', 'remote': True},
    # {'name': 'Logion-BERT', 'path': 'princeton-logion/logion-bert-base', 'type': 'hf', 'remote': True},
    # {'name': 'GreBerta', 'path': 'bowphs/GreBerta', 'type': 'hf', 'remote': True},
    # {'name': 'SPhilBerta-emb-lat-intertext', 'path': 'julian-schelb/SPhilBerta-emb-lat-intertext-v1', 'type': 'hf', 'remote': True},
    # {'name': 'multilingual-e5-base-emb-lat-intertext', 'path': 'julian-schelb/multilingual-e5-base-emb-lat-intertext-v1', 'type': 'hf', 'remote': True},
    # {'name': 'multilingual-e5-base', 'path': 'intfloat/multilingual-e5-base', 'type': 'hf', 'remote': True},
    # {'name': 'LaBSE', 'path': 'sentence-transformers/LaBSE', 'type': 'hf', 'remote': True},
]
for m in MODELS:
    if m.get('remote'):
        print(f"remote model: {m['name']} -> {m['path']}")
        continue
    p = ROOT / m['path']
    assert p.exists(), f"missing: {p}"
    if m['type'] == 'sbert':
        assert (p / 'modules.json').exists(), f"sbert layout missing modules.json in {p}"
    else:
        assert (p / 'config.json').exists(), f"hf layout missing config.json in {p}"
print('all model paths OK')

remote model: LatinBERT -> ashleygong03/bamman-burns-latin-bert
remote model: LaBerta -> bowphs/LaBerta
remote model: SPhilBerta -> bowphs/SPhilBerta
all model paths OK


## 4. STS-based evaluation (reuse detection + IR)

> **Warning — embedding cache:** Cached embeddings are currently keyed only by model name and pooling mode. Delete `results/embeddings/` whenever the input data, text normalization, sequence length, model revision, or model implementation changes; otherwise stale arrays can silently produce invalid results.

We compute the max cosine similarity against all Vulgate verses for each sentence, then:
- detection: AP / PR-AUC + ROC curve on all sentences (reuse vs NaR/NaBR)
- IR: top-1 verse match rate on reuse sentences with gold references

In [12]:
def safe_name(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", name)

def select_device() -> str:
    override = os.getenv("PARAPHRASIS_DEVICE", "").strip().lower()
    if override in ("cpu", "cuda", "auto"):
        if override == "auto" or override == "":
            return "cuda" if torch.cuda.is_available() else "cpu"
        return override
    return "cuda" if torch.cuda.is_available() else "cpu"

def batch_iter(seq, batch_size):
    for i in range(0, len(seq), batch_size):
        yield seq[i : i + batch_size]

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = (token_embeddings * input_mask_expanded).sum(1)
    sum_mask = input_mask_expanded.sum(1).clamp(min=1e-9)
    return sum_embeddings / sum_mask

def resolve_model_path(model_cfg: dict) -> str:
    if model_cfg.get("remote"):
        return model_cfg["path"]
    return str((ROOT / model_cfg["path"]).resolve())

def encode_sbert(texts: List[str], model_path: str, device: str) -> np.ndarray:
    model = SentenceTransformer(model_path, device=device)
    embeddings = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    del model
    torch.cuda.empty_cache()
    return np.asarray(embeddings, dtype="float32")

def encode_hf(texts: List[str], model_path: str, device: str) -> np.ndarray:
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModel.from_pretrained(model_path)
    model.to(device)
    model.eval()
    out = []
    total = (len(texts) + BATCH_SIZE - 1) // BATCH_SIZE
    for batch in tqdm(batch_iter(texts, BATCH_SIZE), total=total, desc="HF encode"):
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LEN,
            return_tensors="pt",
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            output = model(**inputs)
            emb = mean_pooling(output, inputs["attention_mask"])
            emb = F.normalize(emb, p=2, dim=1)
        out.append(emb.cpu().numpy())
    del model, tokenizer
    torch.cuda.empty_cache()
    return np.vstack(out).astype("float32")

# ── Latin BERT (Bamman & Burns) custom tokenizer ─────────────────────────────
# The HF re-upload `ashleygong03/bamman-burns-latin-bert` ships a tensor2tensor
# SubwordTextEncoder vocabulary (word-final pieces carry a trailing "_") but
# wraps it in a BertTokenizer (WordPiece, "##"-continuation). Those two subword
# conventions are incompatible: WordPiece finds no "##" pieces and emits [UNK]
# for ~36% of real Latin tokens, wrecking the embeddings. The class below
# restores the original t2t tokenization (basic tokenizer + "_"-escape + greedy
# longest-match), dropping the [UNK] rate to ~0. Verified on VG.tsv. It returns
# the HF model's own embedding ids, so output feeds straight into AutoModel.
# (CLTK enclitic splitting from Bamman's full pipeline is intentionally omitted;
# the t2t basic tokenizer already handles word/punctuation boundaries.)

class LatinSubwordEncoder:
    SPECIALS = {"[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"}

    def __init__(self, vocab: dict):
        self.subtoken_to_id = {k: v for k, v in vocab.items() if k not in self.SPECIALS}
        self.subtoken_set = set(self.subtoken_to_id)
        self.alphabet = {c for c in self.subtoken_set if len(c) == 1}
        self.max_len = max(len(s) for s in self.subtoken_set)
        self.pad = vocab["[PAD]"]; self.unk = vocab["[UNK]"]
        self.cls = vocab["[CLS]"]; self.sep = vocab["[SEP]"]

    @staticmethod
    def _is_alnum(c: str) -> bool:
        cat = unicodedata.category(c)
        return cat.startswith("L") or cat.startswith("N")

    def _basic_tokenize(self, text: str):
        # t2t basic tokenizer: split at letter/number <-> other transitions,
        # dropping single spaces between word tokens.
        if not text:
            return []
        ret = []; start = 0; isaln = [self._is_alnum(c) for c in text]
        for pos in range(1, len(text)):
            if isaln[pos] != isaln[pos - 1]:
                t = text[start:pos]
                if t != " " or start == 0:
                    ret.append(t)
                start = pos
        ret.append(text[start:])
        return ret

    def _escape(self, token: str) -> str:
        # t2t escaping: "_" marks word end; escape "\", literal "_" and
        # out-of-alphabet chars so greedy match always terminates.
        out = []
        for c in token:
            if c == "\\":
                out.append("\\\\")
            elif c == "_":
                out.append("\\u")
            elif c in self.alphabet and c != "\n":
                out.append(c)
            else:
                out.append("\\%d;" % ord(c))
        return "".join(out) + "_"

    def _subtokenize(self, escaped: str):
        ids = []; start = 0; n = len(escaped)
        while start < n:
            end = min(n, start + self.max_len); found = None
            while end > start:
                sub = escaped[start:end]
                if sub in self.subtoken_set:
                    found = sub; break
                end -= 1
            if found is None:
                return None
            ids.append(self.subtoken_to_id[found]); start = end
        return ids

    def encode(self, text: str, add_special_tokens: bool = True, max_length=None):
        ids = [self.cls] if add_special_tokens else []
        for w in self._basic_tokenize(text.lower()):
            if w == " ":
                continue
            sub = self._subtokenize(self._escape(w))
            ids.extend(sub if sub is not None else [self.unk])
        if add_special_tokens:
            ids.append(self.sep)
        if max_length is not None and len(ids) > max_length:
            if add_special_tokens:
                ids = ids[: max_length - 1] + [self.sep]
            else:
                ids = ids[:max_length]
        return ids


def load_latinbert_vocab(model_path: str) -> dict:
    vocab_path = Path(model_path) / "vocab.txt"
    if not vocab_path.is_file():
        vocab_path = Path(hf_hub_download(model_path, "vocab.txt"))
    with vocab_path.open(encoding="utf-8") as f:
        return {token.rstrip("\r\n"): idx for idx, token in enumerate(f)}


def encode_latinbert(
    texts: List[str], model_path: str, device: str, pooling: str = "mean"
) -> np.ndarray:
    enc = LatinSubwordEncoder(load_latinbert_vocab(model_path))
    config = BertConfig.from_pretrained(model_path)
    model = BertModel.from_pretrained(model_path, config=config)
    model.to(device)
    model.eval()
    out = []
    total = (len(texts) + BATCH_SIZE - 1) // BATCH_SIZE
    for batch in tqdm(batch_iter(texts, BATCH_SIZE), total=total, desc="LatinBERT encode"):
        seqs = [enc.encode(t, add_special_tokens=True, max_length=MAX_SEQ_LEN) for t in batch]
        maxlen = max(len(s) for s in seqs)
        input_ids = torch.full((len(seqs), maxlen), enc.pad, dtype=torch.long)
        attn = torch.zeros((len(seqs), maxlen), dtype=torch.long)
        for i, s in enumerate(seqs):
            input_ids[i, : len(s)] = torch.tensor(s, dtype=torch.long)
            attn[i, : len(s)] = 1
        input_ids = input_ids.to(device); attn = attn.to(device)
        with torch.no_grad():
            output = model(input_ids=input_ids, attention_mask=attn)
            emb = output.last_hidden_state[:, 0] if pooling == "cls" else mean_pooling(output, attn)
            emb = F.normalize(emb, p=2, dim=1)
        out.append(emb.cpu().numpy())
    del model
    torch.cuda.empty_cache()
    return np.vstack(out).astype("float32")



def load_or_encode_embeddings(model_cfg: dict, texts: List[str], cache_path: Path) -> np.ndarray:
    if cache_path.exists():
        return np.load(cache_path)
    model_path = resolve_model_path(model_cfg)
    device = select_device()
    def _encode_for(dev):
        t = model_cfg["type"]
        if t == "sbert":
            return encode_sbert(texts, model_path, dev)
        if t == "latinbert":
            return encode_latinbert(texts, model_path, dev)
        return encode_hf(texts, model_path, dev)
    try:
        emb = _encode_for(device)
    except RuntimeError as e:
        msg = str(e).lower()
        if device == "cuda" and ("no kernel image" in msg or "cuda" in msg):
            print("CUDA error, retrying on CPU")
            emb = _encode_for("cpu")
        else:
            raise
    np.save(cache_path, emb)
    return emb

def build_ip_index(embeddings: np.ndarray) -> faiss.IndexFlatIP:
    emb = embeddings.astype("float32", copy=True)
    faiss.normalize_L2(emb)
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    return index

def search_topk(index: faiss.IndexFlatIP, query_embeddings: np.ndarray, k: int = 1):
    q = query_embeddings.astype("float32", copy=True)
    faiss.normalize_L2(q)
    scores, indices = index.search(q, k)
    return scores, indices


# ── IR ranking metrics: Hits@k and MRR ───────────────────────────────
# Retrieval-with-gold-set metrics for the Vulgate IR task. A query is a "hit@k"
# iff at least one of its gold verses is among the top-k retrieved verses; MRR
# averages the reciprocal rank of the first gold verse (0 when none is in the
# retrieved list). Hits@1 is exactly that top-1 match rate.
HITS_K = [1, 5, 10]
TOPK_IR = max(HITS_K)  # verses to retrieve so every Hits@k / MRR is computable

def compute_ir_ranking_metrics(indices, query_indices, k_values=HITS_K, mrr_cutoff=None):
    """Hits@k and MRR over `query_indices`, given ranked top-k VG `indices`.

    indices: (n_queries, K) array of retrieved VG indices, best first (K >= max k).
    query_indices: row positions to score (e.g. the reuse+gold sentences).
    Returns {"Hits@1": .., "Hits@5": .., "Hits@10": .., "MRR": ..}."""
    if mrr_cutoff is None:
        mrr_cutoff = indices.shape[1]
    denom = len(query_indices)
    if denom == 0:
        return {**{f"Hits@{k}": float("nan") for k in k_values}, "MRR": float("nan")}
    hits = {k: 0 for k in k_values}
    rr_sum = 0.0
    for i in query_indices:
        if short_query_mask[i]:
            continue  # forced not-a-reuse -> miss at every k, no reciprocal rank
        gold = gold_lists[i]
        if not gold:
            continue
        row = indices[i]
        rank = next((r for r, idx in enumerate(row[:mrr_cutoff], start=1) if idx in gold), None)
        if rank is not None:
            rr_sum += 1.0 / rank
        for k in k_values:
            if any(idx in gold for idx in row[:k]):
                hits[k] += 1
    out = {f"Hits@{k}": hits[k] / denom for k in k_values}
    out["MRR"] = rr_sum / denom
    return out


In [13]:
y_true = (~is_nar).astype(int)
has_gold = np.array([len(g) > 0 for g in gold_lists], dtype=bool)
ir_mask = (~is_nar) & has_gold

print(f"Detection dataset: {len(y_true)} sentences | positives={int(y_true.sum())} | negatives={int(is_nar.sum())}")
print(f"IR dataset (reuse + gold): {int(ir_mask.sum())} sentences")

results_detection = []
results_ir = []
roc_curves = []
pr_curves = []

for model_cfg in MODELS:
    if model_cfg.get("remote") and not ALLOW_REMOTE:
        print(f"Skipping remote model: {model_cfg['name']}")
        continue
    name = model_cfg["name"]
    safe = safe_name(name)
    vg_cache = CACHE_DIR / f"{safe}_vg.npy"
    q_cache = CACHE_DIR / f"{safe}_queries.npy"

    print(f"\nModel: {name}")
    vg_emb = load_or_encode_embeddings(model_cfg, vg_texts, vg_cache)
    q_emb = load_or_encode_embeddings(model_cfg, all_texts, q_cache)

    index = build_ip_index(vg_emb)
    scores, indices = search_topk(index, q_emb, k=TOPK_IR)
    max_scores = scores[:, 0]  # reuse probability from max STS
    top1_idx = indices[:, 0]

    # Length filter: force a negative ('not a reuse') prediction for every
    # query with fewer than 5 words. Floor its detection score so it stays
    # below every threshold, and treat its top-1 retrieval as a non-match.
    max_scores = max_scores.copy()
    max_scores[short_query_mask] = -1.0

    # --- Top-100 false positives (mean pooling): NaR sentences with the highest STS ---
    fp_dir = RESULTS_DIR / "false_positives"
    fp_dir.mkdir(exist_ok=True)
    fp_order = [i for i in np.argsort(-max_scores) if is_nar[i]][:100]
    fp_df = pd.DataFrame(
        [
            {
                "reference_a": all_keys[i][1], # index of the query sentence in all_keys
                "reference_b": vg_refs[top1_idx[i]],
                "cosine_similarity": float(max_scores[i]),
                "text_a": all_texts[i],
                "text_b": vg_texts[top1_idx[i]],
            }
            for i in fp_order
        ],
        columns=["reference_a", "reference_b", "cosine_similarity", "text_a", "text_b"],
    )
    fp_path = fp_dir / f"false_positives_{safe}.tsv"
    fp_df.to_csv(fp_path, sep="\t", index=False)
    print(f"Saved {len(fp_df)} false positives -> {fp_path}")

    ap = average_precision_score(y_true, max_scores)
    precision, recall, _ = precision_recall_curve(y_true, max_scores)
    pr_auc = auc(recall, precision)
    fpr, tpr, thresholds = roc_curve(y_true, max_scores)
    roc_auc = auc(fpr, tpr)

    f1_points = []
    for thr in thresholds:
        if np.isinf(thr):
            f1_points.append(0.0)
            continue
        preds = (max_scores >= thr).astype(int)
        tp = np.sum((preds == 1) & (y_true == 1))
        fp = np.sum((preds == 1) & (y_true == 0))
        fn = np.sum((preds == 0) & (y_true == 1))
        precision_thr = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall_thr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision_thr * recall_thr / (precision_thr + recall_thr) if (precision_thr + recall_thr) > 0 else 0.0
        f1_points.append(f1)
    f1_points = np.asarray(f1_points, dtype=float)
    f1_max = float(f1_points.max()) if f1_points.size else float("nan")
    f1_idx = int(f1_points.argmax()) if f1_points.size else 0

    roc_path = RESULTS_DIR / f"roc_{safe}.pdf"
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(fpr, tpr, color="black", linewidth=1, label=f"AUC={roc_auc:.3f}")
    sc = ax.scatter(fpr, tpr, c=f1_points, cmap="viridis", s=12)
    if f1_points.size:
        ax.scatter(fpr[f1_idx], tpr[f1_idx], color="red", s=25, zorder=3, label=f"F1_max={f1_max:.3f}")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC - {name}")
    ax.legend(loc="lower right")
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label("F1")
    fig.tight_layout()
    fig.savefig(roc_path)
    plt.close(fig)

    pr_path = RESULTS_DIR / f"pr_{safe}.pdf"
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(recall, precision, color="black", linewidth=1, label=f"AUC={pr_auc:.3f}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(f"PR - {name}")
    ax.legend(loc="lower left")
    fig.tight_layout()
    fig.savefig(pr_path)
    plt.close(fig)

    roc_curves.append(
        {"name": name, "fpr": fpr, "tpr": tpr, "auc": roc_auc, "f1_max": f1_max, "f1_idx": f1_idx}
    )
    pr_curves.append(
        {"name": name, "recall": recall, "precision": precision, "auc": pr_auc}
    )

    # IR ranking metrics: Hits@k (k=1,5,10) and MRR over reuse+gold sentences.
    ir_query_idx = [i for i in range(len(all_texts)) if ir_mask[i]]
    rank_metrics = compute_ir_ranking_metrics(indices, ir_query_idx)

    results_detection.append(
        {"Model": name, "AP": ap, "ROC_AUC": roc_auc, "F1_max": f1_max}
    )
    results_ir.append(
        {"Model": name, **rank_metrics, "N": len(ir_query_idx)}
    )
    print(f"AP={ap:.3f} ROC-AUC={roc_auc:.3f} F1_max={f1_max:.3f} | "
          f"Hits@1={rank_metrics['Hits@1']:.3f} Hits@10={rank_metrics['Hits@10']:.3f} "
          f"MRR={rank_metrics['MRR']:.3f}")


df_detection = pd.DataFrame(results_detection).sort_values("ROC_AUC", ascending=False).reset_index(drop=True)
df_ir = pd.DataFrame(results_ir).sort_values("MRR", ascending=False).reset_index(drop=True)

det_tex = RESULTS_DIR / "reuse_detection_metrics.tex"
ir_tex = RESULTS_DIR / "reuse_ir_metrics.tex"

df_detection.to_latex(
    det_tex,
    index=False,
    float_format="%.3f",
    caption="Text reuse detection using max STS score per sentence (AP, ROC-AUC, F1_max).",
    label="tab:reuse_detection",
)
df_ir.to_latex(
    ir_tex,
    index=False,
    float_format="%.3f",
    caption="Vulgate IR ranking quality on reuse sentences with gold references "
    "(Hits@$k$ for $k \\in \\{1,5,10\\}$ and MRR; Hits@1 is the top-1 match rate).",
    label="tab:reuse_ir",
)

if roc_curves:
    fig, ax = plt.subplots(figsize=(6, 6))
    for item in roc_curves:
        ax.plot(item["fpr"], item["tpr"], label=f"{item['name']} (AUC={item['auc']:.3f})")
        idx = item["f1_idx"]
        ax.scatter(item["fpr"][idx], item["tpr"][idx], s=18, facecolors="none", edgecolors="black")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC - All Models")
    ax.legend(loc="lower right", fontsize=8)
    fig.tight_layout()
    roc_all_path = RESULTS_DIR / "roc_all_models.pdf"
    fig.savefig(roc_all_path)
    plt.close(fig)
    print("Combined ROC saved:", roc_all_path)

if pr_curves:
    fig, ax = plt.subplots(figsize=(6, 6))
    for item in pr_curves:
        ax.plot(item["recall"], item["precision"], label=f"{item['name']} (AUC={item['auc']:.3f})")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("PR - All Models")
    ax.legend(loc="lower left", fontsize=8)
    fig.tight_layout()
    pr_all_path = RESULTS_DIR / "pr_all_models.pdf"
    fig.savefig(pr_all_path)
    plt.close(fig)
    print("Combined PR saved:", pr_all_path)

display(df_detection)
display(df_ir)
print("Saved LaTeX tables:", det_tex, ir_tex)
print("ROC/PR curves saved in:", RESULTS_DIR)

Detection dataset: 4410 sentences | positives=1196 | negatives=3214
IR dataset (reuse + gold): 1189 sentences

Model: LatinBERT


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17614.20it/s]
[transformers] BertModel LOAD REPORT from: ashleygong03/bamman-burns-latin-bert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14110.06it/s]
[transformers] BertModel LOAD REPORT from: ashleygong03/bamman-burns-latin-bert
Key                  

Saved 100 false positives -> /home/tdelaselle/Documents/BibliReuse/PatriSE/results/false_positives/false_positives_LatinBERT.tsv
AP=0.637 ROC-AUC=0.782 F1_max=0.565 | Hits@1=0.215 Hits@10=0.311 MRR=0.245

Model: LaBerta
Saved 100 false positives -> /home/tdelaselle/Documents/BibliReuse/PatriSE/results/false_positives/false_positives_LaBerta.tsv
AP=0.717 ROC-AUC=0.821 F1_max=0.623 | Hits@1=0.423 Hits@10=0.560 MRR=0.467

Model: SPhilBerta
Saved 100 false positives -> /home/tdelaselle/Documents/BibliReuse/PatriSE/results/false_positives/false_positives_SPhilBerta.tsv
AP=0.621 ROC-AUC=0.772 F1_max=0.550 | Hits@1=0.390 Hits@10=0.556 MRR=0.437
Combined ROC saved: /home/tdelaselle/Documents/BibliReuse/PatriSE/results/roc_all_models.pdf
Combined PR saved: /home/tdelaselle/Documents/BibliReuse/PatriSE/results/pr_all_models.pdf


,Model,AP,ROC_AUC,F1_max
0,LaBerta,0.716764,0.821089,0.623094
1,LatinBERT,0.637461,0.782303,0.564837
2,SPhilBerta,0.621472,0.771737,0.549925


,Model,Hits@1,Hits@5,Hits@10,MRR,N
0,LaBerta,0.423045,0.520606,0.560135,0.467378,1189
1,SPhilBerta,0.390244,0.497897,0.555929,0.436931,1189
2,LatinBERT,0.215307,0.282590,0.311186,0.244611,1189


Saved LaTeX tables: /home/tdelaselle/Documents/BibliReuse/PatriSE/results/reuse_detection_metrics.tex /home/tdelaselle/Documents/BibliReuse/PatriSE/results/reuse_ir_metrics.tex
ROC/PR curves saved in: /home/tdelaselle/Documents/BibliReuse/PatriSE/results


## 4 bis. Same two experiments with **CLS** pooling

Mean-pooling is **not** the universal read-out for sentence encoders — it depends on the training objective:

- **TSDAE** is trained with **CLS pooling**: the decoder reconstructs the sentence from the `[CLS]`/`<s>` vector, so the sentence meaning is concentrated there. Mean-pooling a TSDAE model is *off-objective*.
- **SimCSE / DiffCSE / InfoCSE / RankCSE** follow the SimCSE recipe, which reads out the `[CLS]` token (with an MLP head at training time). They are commonly evaluated on `[CLS]` too.
- Mean-pooling is the safe default mostly for NLI/STS-supervised SBERT-style models.

So evaluating every model with mean-pooling (cell above) can unfairly penalise the CLS-trained ones. The cell below re-runs **exactly** the detection + IR experiments, changing **one thing only — the pooling** — by loading the raw transformer for every model (both `sbert` and `hf`, bypassing any saved pooling module) and reading the `[CLS]`/`<s>` token at position 0. Embedding caches and LaTeX outputs are suffixed `_cls` so they never collide with the mean-pooled results.

In [ ]:
# === CLS-pooling variant of encoding =========================================
# We force [CLS]/<s> read-out on EVERY model (sbert + hf), loading the raw
# transformer and bypassing any saved pooling module, so the only thing that
# differs from the mean-pooling experiment above is the pooling operation.

def cls_pooling(model_output) -> torch.Tensor:
    # position 0 is [CLS] for BERT and <s> for RoBERTa/LaBERTa/GreBERTa
    return model_output.last_hidden_state[:, 0]

def encode_cls(texts: List[str], model_path: str, device: str) -> np.ndarray:
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModel.from_pretrained(model_path)
    model.to(device)
    model.eval()
    out = []
    total = (len(texts) + BATCH_SIZE - 1) // BATCH_SIZE
    for batch in tqdm(batch_iter(texts, BATCH_SIZE), total=total, desc="CLS encode"):
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LEN,
            return_tensors="pt",
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            output = model(**inputs)
            emb = cls_pooling(output)
            emb = F.normalize(emb, p=2, dim=1)
        out.append(emb.cpu().numpy())
    del model, tokenizer
    torch.cuda.empty_cache()
    return np.vstack(out).astype("float32")

def load_or_encode_cls(model_cfg: dict, texts: List[str], cache_path: Path) -> np.ndarray:
    if cache_path.exists():
        return np.load(cache_path)
    model_path = resolve_model_path(model_cfg)
    device = select_device()
    try:
        if model_cfg["type"] == "latinbert":
            emb = encode_latinbert(texts, model_path, device, pooling="cls")
        else:
            emb = encode_cls(texts, model_path, device)
    except RuntimeError as e:
        msg = str(e).lower()
        if device == "cuda" and ("no kernel image" in msg or "cuda" in msg):
            print("CUDA error, retrying on CPU")
            if model_cfg["type"] == "latinbert":
                emb = encode_latinbert(texts, model_path, "cpu", pooling="cls")
            else:
                emb = encode_cls(texts, model_path, "cpu")
        else:
            raise
    np.save(cache_path, emb)
    return emb


# === CLS replication of the two experiments (detection + IR) =================
y_true = (~is_nar).astype(int)
has_gold = np.array([len(g) > 0 for g in gold_lists], dtype=bool)
ir_mask = (~is_nar) & has_gold

print(f"[CLS] Detection dataset: {len(y_true)} sentences | positives={int(y_true.sum())} | negatives={int(is_nar.sum())}")
print(f"[CLS] IR dataset (reuse + gold): {int(ir_mask.sum())} sentences")

results_detection_cls = []
results_ir_cls = []
roc_curves_cls = []
pr_curves_cls = []

for model_cfg in MODELS:
    if model_cfg.get("remote") and not ALLOW_REMOTE:
        print(f"Skipping remote model: {model_cfg['name']}")
        continue
    name = model_cfg["name"]
    safe = safe_name(name)
    vg_cache = CACHE_DIR / f"{safe}_vg_cls.npy"
    q_cache = CACHE_DIR / f"{safe}_queries_cls.npy"

    print(f"\nModel (CLS): {name}")
    vg_emb = load_or_encode_cls(model_cfg, vg_texts, vg_cache)
    q_emb = load_or_encode_cls(model_cfg, all_texts, q_cache)

    index = build_ip_index(vg_emb)
    scores, indices = search_topk(index, q_emb, k=TOPK_IR)
    max_scores = scores[:, 0]  # reuse probability from max STS
    top1_idx = indices[:, 0]

    # Length filter: force a negative ('not a reuse') prediction for every
    # query with fewer than 5 words. Floor its detection score so it stays
    # below every threshold, and treat its top-1 retrieval as a non-match.
    max_scores = max_scores.copy()
    max_scores[short_query_mask] = -1.0

    # --- Top-100 false positives (CLS pooling): NaR sentences with the highest STS ---
    fp_dir = RESULTS_DIR / "false_positives"
    fp_dir.mkdir(exist_ok=True)
    fp_order = [i for i in np.argsort(-max_scores) if is_nar[i]][:100]
    fp_df = pd.DataFrame(
        [
            {
                "sentence_text": all_texts[i],
                "verse_reference": vg_refs[top1_idx[i]],
                "verse_text": vg_texts[top1_idx[i]],
                "sts_score": float(max_scores[i]),
            }
            for i in fp_order
        ],
        columns=["sentence_text", "verse_reference", "verse_text", "sts_score"],
    )
    fp_path = fp_dir / f"false_positives_{safe}_cls.tsv"
    fp_df.to_csv(fp_path, sep="\t", index=False)
    print(f"Saved {len(fp_df)} false positives -> {fp_path}")

    ap = average_precision_score(y_true, max_scores)
    precision, recall, _ = precision_recall_curve(y_true, max_scores)
    pr_auc = auc(recall, precision)
    fpr, tpr, thresholds = roc_curve(y_true, max_scores)
    roc_auc = auc(fpr, tpr)

    f1_points = []
    for thr in thresholds:
        if np.isinf(thr):
            f1_points.append(0.0)
            continue
        preds = (max_scores >= thr).astype(int)
        tp = np.sum((preds == 1) & (y_true == 1))
        fp = np.sum((preds == 1) & (y_true == 0))
        fn = np.sum((preds == 0) & (y_true == 1))
        precision_thr = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall_thr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision_thr * recall_thr / (precision_thr + recall_thr) if (precision_thr + recall_thr) > 0 else 0.0
        f1_points.append(f1)
    f1_points = np.asarray(f1_points, dtype=float)
    f1_max = float(f1_points.max()) if f1_points.size else float("nan")
    f1_idx = int(f1_points.argmax()) if f1_points.size else 0

    roc_path = RESULTS_DIR / f"roc_{safe}_cls.pdf"
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(fpr, tpr, color="black", linewidth=1, label=f"AUC={roc_auc:.3f}")
    sc = ax.scatter(fpr, tpr, c=f1_points, cmap="viridis", s=12)
    if f1_points.size:
        ax.scatter(fpr[f1_idx], tpr[f1_idx], color="red", s=25, zorder=3, label=f"F1_max={f1_max:.3f}")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC (CLS) - {name}")
    ax.legend(loc="lower right")
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label("F1")
    fig.tight_layout()
    fig.savefig(roc_path)
    plt.close(fig)

    pr_path = RESULTS_DIR / f"pr_{safe}_cls.pdf"
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(recall, precision, color="black", linewidth=1, label=f"AUC={pr_auc:.3f}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(f"PR (CLS) - {name}")
    ax.legend(loc="lower left")
    fig.tight_layout()
    fig.savefig(pr_path)
    plt.close(fig)

    roc_curves_cls.append(
        {"name": name, "fpr": fpr, "tpr": tpr, "auc": roc_auc, "f1_max": f1_max, "f1_idx": f1_idx}
    )
    pr_curves_cls.append(
        {"name": name, "recall": recall, "precision": precision, "auc": pr_auc}
    )

    # IR ranking metrics: Hits@k (k=1,5,10) and MRR over reuse+gold sentences.
    ir_query_idx = [i for i in range(len(all_texts)) if ir_mask[i]]
    rank_metrics = compute_ir_ranking_metrics(indices, ir_query_idx)

    results_detection_cls.append(
        {"Model": name, "AP": ap, "ROC_AUC": roc_auc, "F1_max": f1_max}
    )
    results_ir_cls.append(
        {"Model": name, **rank_metrics, "N": len(ir_query_idx)}
    )
    print(f"AP={ap:.3f} ROC-AUC={roc_auc:.3f} F1_max={f1_max:.3f} | "
          f"Hits@1={rank_metrics['Hits@1']:.3f} Hits@10={rank_metrics['Hits@10']:.3f} "
          f"MRR={rank_metrics['MRR']:.3f}")


df_detection_cls = pd.DataFrame(results_detection_cls).sort_values("ROC_AUC", ascending=False).reset_index(drop=True)
df_ir_cls = pd.DataFrame(results_ir_cls).sort_values("MRR", ascending=False).reset_index(drop=True)

det_tex = RESULTS_DIR / "reuse_detection_metrics_cls.tex"
ir_tex = RESULTS_DIR / "reuse_ir_metrics_cls.tex"

df_detection_cls.to_latex(
    det_tex,
    index=False,
    float_format="%.3f",
    caption="Text reuse detection with CLS pooling using max STS score per sentence (AP, ROC-AUC, F1_max).",
    label="tab:reuse_detection_cls",
)
df_ir_cls.to_latex(
    ir_tex,
    index=False,
    float_format="%.3f",
    caption="Vulgate IR ranking quality with CLS pooling on reuse sentences with gold "
    "references (Hits@$k$ for $k \\in \\{1,5,10\\}$ and MRR; Hits@1 is the top-1 match rate).",
    label="tab:reuse_ir_cls",
)

if roc_curves_cls:
    fig, ax = plt.subplots(figsize=(6, 6))
    for item in roc_curves_cls:
        ax.plot(item["fpr"], item["tpr"], label=f"{item['name']} (AUC={item['auc']:.3f})")
        idx = item["f1_idx"]
        ax.scatter(item["fpr"][idx], item["tpr"][idx], s=18, facecolors="none", edgecolors="black")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC (CLS) - All Models")
    ax.legend(loc="lower right", fontsize=8)
    fig.tight_layout()
    roc_all_path = RESULTS_DIR / "roc_all_models_cls.pdf"
    fig.savefig(roc_all_path)
    plt.close(fig)
    print("Combined ROC (CLS) saved:", roc_all_path)

if pr_curves_cls:
    fig, ax = plt.subplots(figsize=(6, 6))
    for item in pr_curves_cls:
        ax.plot(item["recall"], item["precision"], label=f"{item['name']} (AUC={item['auc']:.3f})")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("PR (CLS) - All Models")
    ax.legend(loc="lower left", fontsize=8)
    fig.tight_layout()
    pr_all_path = RESULTS_DIR / "pr_all_models_cls.pdf"
    fig.savefig(pr_all_path)
    plt.close(fig)
    print("Combined PR (CLS) saved:", pr_all_path)

display(df_detection_cls)
display(df_ir_cls)
print("Saved LaTeX tables (CLS):", det_tex, ir_tex)
print("ROC/PR curves (CLS) saved in:", RESULTS_DIR)
